In [ ]:
import pandas as pd
import random
import numpy as np
# import torch
# from transformers import AutoTokenizer, AutoModel, BertConfig, BertModel
# import adapters
# from adapters import AutoAdapterModel
# import gc
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import matplotlib

import pickle
import time
import memory_profiler

%load_ext memory_profiler

from pathlib import Path
import distro

%load_ext watermark

In [ ]:
# %load_ext IPython.extensions.autoreload
# %autoreload 2

# from src.model_stuff import (
#     fix_all_seeds,
# )

In [ ]:
import jupyter_black

jupyter_black.load(line_length=79)

In [ ]:
variables_path = Path("../results/variables")
figures_path = Path("../results/figures/tmp")
data_path = Path("../data")

berenslab_data_path = Path("/gpfs01/berens/data/data/pubmed_processed")

In [ ]:
pwd

'/gpfs01/berens/user/rgonzalesmarquez'

In [ ]:
# MANUAL FIX TO PATH ISSUE FROM VSCODE

nb_path = Path(
    "/gpfs01/berens/user/rgonzalesmarquez/phd/pubmed-retina/scripts"
)
assert nb_path.exists(), "The path does not exist"

variables_path = (nb_path / variables_path).resolve(strict=True)
figures_path = (nb_path / figures_path).resolve(strict=True)
data_path = (nb_path / data_path).resolve(strict=True)

In [ ]:
plt.style.use(nb_path / "matplotlib_style.txt")

In [ ]:
%watermark -a 'Rita González-Márquez' -t -d -tz -u -v -iv -w -m -h -p transformers -p openTSNE
print(distro.name(pretty=True))

Author: Rita González-Márquez

Last updated: 2025-05-19 10:14:26CEST

Python implementation: CPython
Python version       : 3.11.5
IPython version      : 8.18.1

openTSNE: 1.0.0

Compiler    : GCC 11.2.0
OS          : Linux
Release     : 3.10.0-1160.el7.x86_64
Machine     : x86_64
Processor   : x86_64
CPU cores   : 40
Architecture: 64bit

Hostname: rgonzalesmarquez_GPU0-llm_gber3

memory_profiler: 0.61.0
jupyter_black  : 0.3.4
numpy          : 1.26.2
matplotlib     : 3.8.2
distro         : 1.8.0
pandas         : 2.1.3

Watermark: 2.4.3

Ubuntu 22.04.3 LTS


# Import

In [ ]:
%%time
df_2025 = pd.read_parquet(
    berenslab_data_path / "pubmed_baseline_2025.parquet.gzip",
    engine="pyarrow",
)

CPU times: user 3min 49s, sys: 1min, total: 4min 49s
Wall time: 2min 48s


In [ ]:
df_2025.head()

,PMID,Title,AbstractText,Journal,Year,Labels,Countries,InferredGenderFirstAuthor,InferredGenderLastAuthor
0,1000,The amino acid sequence of Neurospora NADP-spe...,The NADP-specific glutamate dehydrogenase of N...,The Biochemical journal,1975,unlabeled,unknown,unknown,unknown
1,10000,A new method for the determination of alpha1-p...,Up until now it has been assumed that the prot...,Biochimica et biophysica acta,1976,unlabeled,unknown,unknown,unknown
2,1000017,Development of a short term assay for lymphobl...,Blastic stimulation of lymphocytes by antigens...,Biomedicine / [publiee pour l'A.A.I.C.I.G.],1976,unlabeled,unknown,unknown,unknown
3,1000018,Neonatal polycythemia in low birthweight infant.,74 low birthweight infants haematologic findin...,Biomedicine / [publiee pour l'A.A.I.C.I.G.],1976,unlabeled,unknown,unknown,unknown
4,1000019,Is there evidence for subclasses of chronic ly...,Various clinical and biological parameters wer...,Biomedicine / [publiee pour l'A.A.I.C.I.G.],1976,unlabeled,unknown,unknown,unknown


In [ ]:
df_2025.shape

(24814136, 9)

In [ ]:
type(df_2025.PMID.iloc[0])

str

In [ ]:
%%time
df_2024 = pd.read_csv(
    berenslab_data_path / "pubmed_landscape_data_2024_v2.zip",
    engine="pyarrow",
)

CPU times: user 1min 58s, sys: 12.3 s, total: 2min 10s
Wall time: 1min 16s


In [ ]:
df_2024.head()

,Title,Journal,PMID,Year,x,y,Labels,Colors,Retractions,Countries,InferredGenderFirstAuthor,InferredGenderLastAuthor
0,Influence of a new virostatic compound on the ...,Arzneimittel-Forschung,24,1975,-37.297431,12.397200,unlabeled,#D3D3D3,False,unknown,unknown,unknown
1,Effect of etafenone on total and regional myoc...,Arzneimittel-Forschung,23,1975,8.703783,32.313807,unlabeled,#D3D3D3,False,unknown,unknown,unknown
2,Pharmacological properties of new neuroleptic ...,Arzneimittel-Forschung,25,1975,-139.371149,-18.888539,unlabeled,#D3D3D3,False,unknown,unknown,unknown
3,Lysosomal hydrolases of the epidermis. I. Glyc...,The British journal of dermatology,30,1975,-80.688904,2.023305,dermatology,#D790FF,False,unknown,unknown,unknown
4,A serum haemagglutinating property dependent u...,British journal of haematology,32,1975,-81.988885,-15.153657,unlabeled,#D3D3D3,False,unknown,unknown,unknown


In [ ]:
df_2024.shape

(23389083, 12)

In [ ]:
type(df_2024.PMID.iloc[0])

numpy.int64

In [ ]:
df_2024["PMID"] = df_2024["PMID"].astype(str)

In [ ]:
%%time
filtered_df_2025 = df_2025[~df_2025["PMID"].isin(df_2024["PMID"])]

CPU times: user 16.9 s, sys: 983 ms, total: 17.9 s
Wall time: 17.6 s


In [ ]:
filtered_df_2025.shape[0]

1430687

In [ ]:
filtered_df_2025

,PMID,Title,AbstractText,Journal,Year,Labels,Countries,InferredGenderFirstAuthor,InferredGenderLastAuthor
7557590,22431807,Mucormycosis in organ and stem cell transplant...,Mucormycosis is a devastating invasive fungal ...,Clinical infectious diseases : an official pub...,2012,infectious,France,female,male
8531228,23680746,Reactions to institutional violence: patient s...,In this article we identify evidences of inequ...,Salud colectiva,2013,unlabeled,unknown,female,female
8531229,23680747,Old age in primary school readers: a journey t...,This article presents the content (discourse) ...,Salud colectiva,2013,unlabeled,Argentina,unknown,unknown
8531230,23680748,Treatment for cancer pain at the end of life: ...,Cancer pain relief has been defined as a world...,Salud colectiva,2013,unlabeled,Argentina,male,male
8531231,23680749,The origin and quality of water for human cons...,The aim of this study is to analyze the origin...,Salud colectiva,2013,unlabeled,Argentina,female,female
...,...,...,...,...,...,...,...,...,...
22246941,39776041,Assessment of ICG fluorescence in identificati...,Intraoperative parathyroid gland (PG) localiza...,Endocrine,2025,unlabeled,India,unknown,unknown
22246942,39776042,Functional study of three cases with novel TBX...,Congenital isolated adrenocorticotropic hormon...,Endocrine,2025,unlabeled,China,unknown,unknown
22246943,39776043,"The Patterns of P53, E-Cadherin, β-Catenin, CX...",Oral squamous cell carcinoma (OSCC) is a signi...,Head and neck pathology,2025,pathology,Brazil,female,male
22246944,39776044,Deep Learning-Enabled Automated Quality Contro...,Several factors can impair image quality and r...,Journal of magnetic resonance imaging : JMRI,2025,unlabeled,United States,male,unknown


In [ ]:
df_2024.shape[0] + filtered_df_2025.shape[0]

24819770

In [ ]:
filtered_df_2025_2 = df_2025.groupby(["PMID"]).last()

In [ ]:
df_2025_2.shape

(24814136, 8)

In [ ]:
df_2025_2 = df_2025.groupby(["PMID"]).last()

In [ ]:
df_2025_2.shape 

NameError: name 'df_2025_2' is not defined

In [ ]:
24814136

In [ ]:
import gc


# Delete the original dataframes
del df_2024
del df_2025

# Optionally, force garbage collection
gc.collect()

3075

# Test

In [ ]:
# DataFrame 1 (df1) - contains some IDs
df1 = pd.DataFrame(
    {
        "ID": ["1", "2", "2", "3", "4", "5"],
        "Value1": ["A", "B", "B", "C", "D", "E"],
    }
)

# DataFrame 2 (df2) - contains some overlapping and some unique IDs
df2 = pd.DataFrame(
    {
        "ID": ["3", "4", "5", "6", "7", "8"],
        "Value2": ["X", "Y", "Z", "W", "Q", "R"],
    }
)

print("df1:")
print(df1)
print("\ndf2:")
print(df2)

df1:
  ID Value1
0  1      A
1  2      B
2  2      B
3  3      C
4  4      D
5  5      E

df2:
  ID Value2
0  3      X
1  4      Y
2  5      Z
3  6      W
4  7      Q
5  8      R


In [ ]:
%%time
%%memit
filtered_df2 = df2[~df2["ID"].isin(df1["ID"])]
print("\nEntries in df2 with IDs NOT in df1 (using isin()):")
print(filtered_df2)


Entries in df2 with IDs NOT in df1 (using isin()):
  ID Value2
3  6      W
4  7      Q
5  8      R
peak memory: 72222.96 MiB, increment: 0.00 MiB
CPU times: user 154 ms, sys: 655 ms, total: 809 ms
Wall time: 1.42 s


In [ ]:
%%time
%%memit
merged = df2.merge(df1[["ID"]], on="ID", how="left", indicator=True)
filtered_df2_merge = merged[merged["_merge"] == "left_only"]
print("\nEntries in df2 with IDs NOT in df1 (using merge()):")
print(filtered_df2_merge)


Entries in df2 with IDs NOT in df1 (using merge()):
  ID Value2     _merge
3  6      W  left_only
4  7      Q  left_only
5  8      R  left_only
peak memory: 198.33 MiB, increment: 0.01 MiB
CPU times: user 185 ms, sys: 96 ms, total: 281 ms
Wall time: 389 ms


# stack

In [ ]:
import numpy as np
a = np.zeros((2,3))
print(a)

[[0. 0. 0.]
 [0. 0. 0.]]


In [ ]:
np.vstack((a,a))

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

# core dump

In [ ]:
saving_path = Path("embeddings/2025_baseline/difference")

In [ ]:
print(np.load(berenslab_data_path / saving_path / "last_i_batch.npy"))

79600


In [ ]:
embedding_sep_interm = np.load(
    berenslab_data_path / saving_path / "embedding_sep_interm.npy"
)

In [ ]:
embedding_sep_interm.shape

(20326656, 768)

In [ ]:
a = np.zeros((11, 2))
print(a.shape)
aa = np.vstack((a, a))
print(aa.shape)

(11, 2)
(22, 2)


In [ ]:
aa[11:].shape

(11, 2)

In [ ]:
aa_test = np.vstack((aa[:11], aa[11:]))
aa_test.shape

(22, 2)

In [ ]:
aa_df = pd.DataFrame({"a": aa[:, 0]})

In [ ]:
print(aa_df)
aa_df.a.iloc[11:] == aa_df.a[11:]

      a
0   0.0
1   0.0
2   0.0
3   0.0
4   0.0
5   0.0
6   0.0
7   0.0
8   0.0
9   0.0
10  0.0
11  0.0
12  0.0
13  0.0
14  0.0
15  0.0
16  0.0
17  0.0
18  0.0
19  0.0
20  0.0
21  0.0


11    True
12    True
13    True
14    True
15    True
16    True
17    True
18    True
19    True
20    True
21    True
Name: a, dtype: bool